
# **SEMESTER PROJECT DELIVERABLE 2 (ML ENGINE)**

## **Group:** 4

## **Members:**

##### 191231 - Patrick Mungai

##### 192082 - DavidTelios Muriithi


## **Model:** Logistic Regression

## **Task:** Resume to job-category recommendation (multi-class text classification used for top-k ranking).

## **Dataset:** Kaggle Resume Dataset - [https://www.kaggle.com/datasets/snehaanbhawal/resume-dataset]()


### 1. SETUP


In [2]:
import re, warnings # For finding patterns in text
from pathlib import Path
import numpy as np # For numbers
import pandas as pd # For tables
import matplotlib.pyplot as plt # For charts

warnings.filterwarnings("ignore")
RANDOM_STATE = 42 # Fixes any random choices
matches = list(Path(".").rglob("Resume.csv"))
if not matches:
    raise FileNotFoundError("Resume.csv not found. Download the Kaggle Resume Dataset (link in README), unzip it, and place Resume.csv inside the 'Resume' folder.")
CSV_PATH = matches[0]
print("Using:", CSV_PATH)

Using: Resume\Resume.csv


### 2. LOADING & LOOKING

For this, the category field is our "answer" and the resume is like our "question". The model should be able to identify where each resume fits into which category.

There are no missing values found, but that only means no blank cells in the dataset.

In [3]:
raw = pd.read_csv(CSV_PATH)
print("Shape:", raw.shape)
print(raw.dtypes, "\n")
print("Missing values:\n", raw.isnull().sum(), "\n")
print("Number of classes:", raw["Category"].nunique())
raw.head(3)

Shape: (2484, 4)
ID             int64
Resume_str       str
Resume_html      str
Category         str
dtype: object 

Missing values:
 ID             0
Resume_str     0
Resume_html    0
Category       0
dtype: int64 

Number of classes: 24


,ID,Resume_str,Resume_html,Category
0,16852973,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,"<div class=""fontsize fontface vmargins hmargin...",HR
1,22323967,"HR SPECIALIST, US HR OPERATIONS ...","<div class=""fontsize fontface vmargins hmargin...",HR
2,33176873,HR DIRECTOR Summary Over 2...,"<div class=""fontsize fontface vmargins hmargin...",HR


### 3. DETECTING PROBLEMS

We measured each possible problem first using the data cleaning process

| Problem | How we checked | What we found | Why it matters |
| --- |--------------------------------| --- | --- |
| Empty resume | Text under 200 characters      | 1 | Nothing from the model to learn from |
| Duplicates | Identical resume text          | 2 | A copy could land in both training and test sets and inflate our score |
| Placeholder text "Company Name" | Searched for it                | 2483 | Appears in almost all resumes, hence meaning it carries no information |
| Encoding junk (eg Â, â€) | Searched for broken characters | 3 | These characters are meaningless |
| Job title at the start | Read samples                   | Present | When title is near the label, the model could cheat instead of learning |
| Different resume lengths | Shortest and longest           | 21 to 38,842 characters | Long resumes dominate word count unless adjusted |


### Unexpected problems
We read random resumes, counted unusual characters, and listed words that appear in nearly every resume. This found:
- **Label noise:** eg. Resume 940 is labelled AGRICULTURE but describes a bank examiner. This limits how accurate the model can be.
- **Unusual characters:** a fullwidth hyphen, non-breaking spaces ([866]) and invisible zero-width spaces ([166]).
- **Personal data:** emails ([19]), web links ([98]), and details like marital status ([66]). We remove them.
- **Template leftovers:** text like "[positive outcome]" ([176]) left by resume-writing tools.
- **Spacing:** When searching for like a City or State with exact spacing found 0 data, the text uses irregular spacing hence the model has to employ flexible spacing.

In [10]:
df = raw.copy()
text = df["Resume_str"].fillna("")

print("Empty/near-empty (<=200 chars):", (text.str.strip().str.len()<= 200).sum())
print("Exact duplicate texts:", text.duplicated().sum())
print("Contain 'Company Name' placeholder:", text.str.contains("Company Name").sum())
print("Contain encoding junk:", text.str.contains("Â|ï¼|â€|Ã", regex = True).sum())
print("Shortest lengths:", sorted(text.str.len())[:5])
print("Longest length:", text.str.len().max())
print("\nHow a resume opens:\n", text.iloc[0][:150])

Empty/near-empty (<=200 chars): 1
Exact duplicate texts: 2
Contain 'Company Name' placeholder: 2483
Contain encoding junk: 3
Shortest lengths: [21, 881, 985, 995, 1033]
Longest length: 38842

How a resume opens:
          HR ADMINISTRATOR/MARKETING ASSOCIATE

HR ADMINISTRATOR       Summary     Dedicated Customer Service Manager with 15+ years of experience in H


### 4. DATA CLEANING

In [5]:
df = raw.copy() # Using a copy of the data
log = [] # Logs every step done in a table
def step(name, before, after):
    log.append({"step": name, "rows_before": before, "rows_after": after})

# Drop the redundant HTML column since it repeats Resume_str with web formatting
df = df.drop(columns=["Resume_html"])

# Drop empty resumes
n = len(df)
df["Resume_str"] = df["Resume_str"].fillna("")
df = df[df["Resume_str"].str.strip().str.len() > 200]
step(" Drop empty (<=200 chars)", n, len(df))

# Drop duplicate texts
n = len(df)
df = df.drop_duplicates(subset="Resume_str", keep="first")
step("Drop duplicates", n, len(df))

# Keeping the original for EDA
df["raw_text"] = df["Resume_str"]
df = df.reset_index(drop = True)
pd.DataFrame(log)

,step,rows_before,rows_after
0,Drop empty (<=200 chars),2484,2483
1,Drop duplicates,2483,2481


In [11]:
import unicodedata


def fix_encoding_junk(t):
    # Replacing bad characters with a space
    for bad in ["Â", "ï¼", "â€“", "â€™", "Ã"]:
        t = t.replace(bad, " ")
    return t

def strip_title(t):
    # Removes all capital letters at start of resumes since they open with job title
    m = re.match(r"^\s*(?:[A-Z0-9/][A-Z0-9/&,\-\.\(\)']*\s+)+", t)
    s = t[m.end():] if m else t
    return s if len(s.strip()) > 200 else t

def clean(t):
    t = strip_title(t) # Removes the leading Job title
    t = fix_encoding_junk(t) # Replaces the bad characters
    t = t.replace("Company Name", " ").replace("City, State", " ")
    t = re.sub(r"[\w\.-]+@[\w\.-]+\.\w+", " ", t) # 3. remove email addresses (privacy)
    t = re.sub(r"https?://\S+|www\.\S+", " ", t) # Removes web links
    t = unicodedata.normalize("NFKD", t).encode("ascii", "ignore").decode()  # Turns odd characters into plain ones (fullwidth hyphen to "-", é to e) and drops invisible ones
    t = re.sub(r"\[[^\]]{3,40}\]", " ", t) # Removes template text like [positive outcome]
    t = re.sub(r"[^A-Za-z+#\s]", " ", t) # Keeps letters, + and #
    t = re.sub(r"\s+", " ", t).strip().lower() # Keeps one space between words and letters to lowercase
    return t

df["text"] = df["Resume_str"].apply(clean)
n = len(df)
df = df[df["text"].str.len() > 100].reset_index(drop = True)
step("Drop resumes empty after cleaning (<=100 chars)", n, len(df))
pd.DataFrame(log)


,step,rows_before,rows_after
0,Drop empty (<=200 chars),2484,2483
1,Drop duplicates,2483,2481
2,Drop resumes empty after cleaning (<=100 chars),2481,2481
3,Drop resumes empty after cleaning (<=100 chars),2481,2481
4,Drop resumes empty after cleaning (<=100 chars),2481,2481
5,Drop resumes empty after cleaning (<=100 chars),2484,2483
